- Title page
- Table of Contents
- Problem statement and its short explanation of the goal
- Methodology / problem-solving approach
- Key points of your technique/algorithms/ideas for speed-up
- Limitations and challenges encountered
- Test/example functions used, with graphical visualizations and output results
- Computational cost and time-complexity analysis (if you can)
- Ideas for further improvements of the projects
- References

In [26]:
import numpy as np
import torch
from mytorch.tensor.arithmetic import *
from mytorch.tensor.arithmetic import MyConv2d

## Tensor

## Optimization
### Stochastic Gradient Descent
### Adaptive Moment Estimation
### Adaptive Gradient Algorithm
### Optimization Strategy
After having all the optimization algorithms to choose from, there are still three different strategies that you can choose from to update our model.
#### Stochastic Optimization
Stochastic optimization means that we feed the data into the model **one sample at a time** and update the model immediately. The term stochastic indicates the randomness involved in how the updates are computed. This strategy can be formulated as:
$$
\theta \leftarrow \theta - \eta \nabla_\theta L\big(f_\theta(x_i),\, y_i\big)
$$
- $f_\theta$: The model with parameters $\theta$
- $\eta$: The learning rate
- $L$: The loss function
- $x_i$: The $i$th input sample
- $y_i$: The $i$th ground truth label

However, it has some major disadvantages that make it no longer an ideal strategy:
- One single sample is not representative enough and too noisy, the computed gradients might lead to wrong updates.
- Unstable updates can cause the model to miss the global optimum, which leads to poor convergence.
- Using one sample at a time doesn't make good use of GPU, which leads to slow computation.

#### Batch Optimization
Batch optimization means that the entire dataset is fed into the model and update the model based on the sum or the mean of the losses. This can be formulated as:
$$
\theta \leftarrow \theta - \eta \nabla_\theta \frac{1}{N} \sum_{i=1}^{N} L\big(f_\theta(x_i),\, y_i\big)
$$
- $N$: The size of the entire dataset

Similarly, it also has some disadvantages:
- Training can be extremely slow for very large datasets like [ImageNet](https://www.image-net.org/).
- High memory requirement.
- Tend to stuck in local optimum as no randomness is involved.

#### Mini-batch Optimization
Mini-batch optimization is a trade-off between the above two strategies. We feed a part of the dataset into the model. This can be formulated as:
$$
\theta \leftarrow \theta - \eta \nabla_\theta \frac{1}{B} \sum_{i=1}^{B} L\big(f_\theta(x_i),\, y_i\big)
$$
- $B$: The batch size

It is important to choose a right batch size when using this strategy.

## Convolutional Neural Network
[Convolutional neural network](https://en.wikipedia.org/wiki/Convolutional_neural_network) (CNN or ConvNet) is a well-established deep learning model. It specializes in processing spatially structured data like images. Its most notable feature is its use of convolutional kernels or filters. Kernels carry the trainable weights of the model, they go over the images and generates feature maps that capture some certain features of the original images, and they will be optimized during the training.

This design is actually inspired by the human visual system, each kernel in CNN serves as a visual neuron, while every neuron can only detect a small part of the visual field that is its [receptive field](https://en.wikipedia.org/wiki/Receptive_field). Furthermore, neurophysiologists [David H. Hubel](https://en.wikipedia.org/wiki/David_H._Hubel) and [Torsten Wiesel](https://en.wikipedia.org/wiki/Torsten_Wiesel) have revealed that human visual system is hierarchical, low-level neurons detect simple features like edges, lines or circles, while higher-level neurons integrate the information in order to recognize more complex patterns like cars, houses or people.

Compare to other deep learning model, CNN is known for its shift invariance, which means it doesn't matter where a certain pattern like a car appears in the image, it is able to recognize it. Another advantage of CNN is that it often requires fewer parameters due to its shared-weight architecture as the same kernel is reused for the whole image, unlike linear model that every pixel of the image is assigned with a unique parameter, which can lead to a very high memory-cost and problems like vanishing gradients or exploding gradients. For example, consider an input image of size 100 × 100 with 3 channels (RGB), each parameter of the next layer requires 100 × 100 × 3 = 30,000 weights in any fully connected neural network, while it only needs 10 × 10 × 3 = 300 weights in a CNN with a kernel size of 10 × 10.

In this chapter we will cover the following topics:
- Introduction of 2D convolution operation
- Algorithms and some linear algebra used in the forward pass
- Algorithms and some simple matrix calculus used in the backward pass
- Training on [MNIST](https://en.wikipedia.org/wiki/MNIST_database) dataset

### 2D Convolution Operation
In functional analysis, [convolution](https://en.wikipedia.org/wiki/Convolution) is a mathematical operation that takes two functions $f(x)$ and $g(x)$, and produces a new function $(f * g)(x)$ by first reflecting one of the functions over the y-axis, then sliding it over the other function, summing the products of their values at each point where they overlap. For example, reflecting $g(x)$ gives $g(-x)$. Then, sliding $g(-x)$ over $f(x)$ by a displacement of $t$ and multiplying their values gives $f(x)g(t - x)$. Finally, summing the products by taking the integral gives us its definition:
$$
(f * g)(t) := \int_{-\infty}^\infty f(\tau) g(t - \tau) \, d\tau.
$$

However, in the field of deep learning, the term convolution is often used to refer to [cross-correlation](https://en.wikipedia.org/wiki/Cross-correlation). For example, CNN actually uses cross-correlation instead of convolution. although the two operations are very similar, the only difference is that cross-correlation does not involve function reflection. For continuous functions $f$ and $g$, cross-correlation is defined as:
$$
(f \star g)(t) := \int_{-\infty}^\infty f(\tau) g(t + \tau) \, d\tau.
$$

For consistency, we will continue to use convolution to refer to the operation used in CNN. Furthermore, since images and kernels can be viewed as discrete functions in 2D, the discrete 2D convolution is defined as:
$$
(I * K)(i, j) := \sum_{p=0}^{H_{\text{k}}-1} \sum_{q=0}^{W_{\text{k}}-1} I(i + p,\, j + q) K(p,\, q).
$$
- $H_{\text{k}}$: The height of the kernel
- $W_{\text{k}}$: The width of the kernel
- $I(i,\, j)$: The value of pixel $(i,\, j)$ in the image
- $K(p,\, q)$: The value of parameter $(p,\, q)$ in the kernel

2D convolution is the core mathematical operation in CNN. The kernel acts as a sliding window over the image, summing the element-wise products between the kernel and the corresponding pixels of the image, like taking a dot product after they are both vectorized. For example, given an image of shape $(5,\, 5)$ and a kernel of shape $(3,\, 3)$:
$$
I =
\begin{bmatrix}
0 & 1 & 1 & 1 & 0 \\
0 & 1 & 0 & 1 & 0 \\
0 & 1 & 1 & 1 & 0 \\
0 & 0 & 0 & 1 & 0 \\
0 & 1 & 1 & 1 & 0
\end{bmatrix},
\quad
K =
\begin{bmatrix}
1 & 2 & 3 \\
4 & 5 & 6 \\
7 & 8 & 9
\end{bmatrix}
$$

As you can see, the tiny binary image resembles a handwritten digit $9$, while the kernel is randomly initialized with natural numbers. According to the equation above, we have:
$$
\begin{align}
(I * K)(0,\, 0) &=
\operatorname{vec}\left(
\begin{bmatrix}
0 & 1 & 1 \\
0 & 1 & 0 \\
0 & 1 & 1
\end{bmatrix}
\right)
\cdot
\operatorname{vec}\left(
\begin{bmatrix}
1 & 2 & 3 \\
4 & 5 & 6 \\
7 & 8 & 9
\end{bmatrix}
\right) \\
&= \sum_{p=0}^{2} \sum_{q=0}^{2} I(p,\, q) K(p,\, q) \\
&= 27
\end{align}
$$

This is achieved simply by placing the kernel over the top-left corner of the image and calculating the dot product of their vectorized forms. Similarly, we can compute the value of $(I * K)(1,\, 1)$ by sliding the kernel to the right by one column and down by one row, we have:
$$
\begin{align}
(I * K)(1,\, 1) &=
\operatorname{vec}\left(
\begin{bmatrix}
1 & 0 & 1 \\
1 & 1 & 1 \\
0 & 0 & 1
\end{bmatrix}
\right)
\cdot
\operatorname{vec}\left(
\begin{bmatrix}
1 & 2 & 3 \\
4 & 5 & 6 \\
7 & 8 & 9
\end{bmatrix}
\right) \\
&= \sum_{p=0}^{2} \sum_{q=0}^{2} I(1 + p,\, 1 + q) K(p,\, q) \\
&= 28
\end{align}
$$

Finally, we can obtain the entire output of shape $(3,\, 3)$ as follows:
$$
I * K =
\begin{bmatrix}
27 & 40 & 23 \\
13 & 28 & 19 \\
22 & 36 & 23
\end{bmatrix}
$$


### Forward Pass in CNN
The forward pass in CNN essentially performs a 2D convolution. So far, the images we have used are 2D matrices of shape $(H,\, W)$. However, in real-world scenarios, input image is usually a 3D tensors of shape $(C_{\text{in}},\, H,\, W)$, where $C_{\text{in}}$ denotes the number of input channels. For example, a standard RGB image has three channels corresponding to red, green and blue. Therefore, the kernel must also be expanded from $(H_{\text{k}},\, W_{\text{k}})$ to shape $(C_{\text{in}},\, H_{\text{k}},\, W_{\text{k}})$ to match the channel of the input image. One kernel produces one output channel, so in order to generate a multichannel output, multiple kernels are used. In practice, multiple kernels form a 4D tensor of shape $(C_{\text{out}},\, C_{\text{in}},\, H_{\text{k}},\, W_{\text{k}})$, where $C_{\text{out}}$ denotes the number of output channels, and batched input images form a 4D tensor of shape $(B,\, C_{\text{in}},\, H,\, W)$, where $B$ denotes the batch size. The 2D convolution is then computed as:
$$
(I * K)(b,\, c_{\text{out}},\, i,\, j) =
\sum_{c_{\text{in}}=0}^{C_{\text{in}}-1}
\sum_{p=0}^{H_{\text{k}}-1}
\sum_{q=0}^{W_{\text{k}}-1}
I(b,\, c_{\text{in}},\, i + p,\, j + q)\,
K(c_{\text{out}},\, c_{\text{in}},\, p,\, q)
$$

Furthermore, there are three main hyperparameters that control the behavior of a CNN:
- Padding
- Dilation
- stride

Padding determines the number of additional rows and columns added around the original image, usually filled with zeros. It can enable kernels to capture more information near the image boundaries. For a padding of $(P_{\text{h}},\, P_{\text{w}})$ and an image of shape $(H,\, W)$, $P_{\text{h}}$ rows are added to both the top and bottom of the image, and $P_{\text{w}}$ columns are added to both the left and right sides of the image, producing a padded image of shape $(H + 2P_{\text{h}},\, W + 2P_{\text{w}})$. For example, if we apply a padding of $(1,\, 1)$ to the image $I$ shown above, we will get an $I_{\text{padded}}$ of shape $(7,\, 7)$:
$$
I_{\text{padded}} =
\begin{bmatrix}
0 & 0 & 0 & 0 & 0 & 0 & 0 \\
0 & 0 & 1 & 1 & 1 & 0 & 0 \\
0 & 0 & 1 & 0 & 1 & 0 & 0 \\
0 & 0 & 1 & 1 & 1 & 0 & 0 \\
0 & 0 & 0 & 0 & 1 & 0 & 0 \\
0 & 0 & 1 & 1 & 1 & 0 & 0 \\
0 & 0 & 0 & 0 & 0 & 0 & 0
\end{bmatrix}
$$

Dilation determines the number of zeros inserted between neighboring parameters of the original kernel. It can effectively enlarge the receptive field without increasing the number of parameters. For a dilation of $(D_{\text{h}},\, D_{\text{w}})$ and a kernel of shape $(H_{\text{k}},\, W_{\text{k}})$, $(D_{\text{h}} - 1)$ zeros are inserted between neighboring parameters in each column, and $(D_{\text{w}} - 1)$ zeros are inserted between neighboring parameters in each row, producing a dilated kernel of shape $(D_{\text{h}} H_{\text{k}} - D_{\text{h}} + 1,\, D_{\text{w}} W_{\text{k}} - D_{\text{w}} + 1)$. For example, if we apply a dilation of $(2,\, 2)$ to the kernel $K$ shown above, we will get a $K_{\text{dilated}}$ of shape $(5,\, 5)$:
$$
K_{\text{dilated}} =
\begin{bmatrix}
1 & 0 & 2 & 0 & 3 \\
0 & 0 & 0 & 0 & 0 \\
4 & 0 & 5 & 0 & 6 \\
0 & 0 & 0 & 0 & 0 \\
7 & 0 & 8 & 0 & 9
\end{bmatrix}
$$

Finally, stride determines how the kernel moves over the image. For an arbitrary stride of $(S_{\text{h}}, S_{\text{w}})$, giving padding and dilation, the 2D convolution is then given by the following equation:
$$
(I * K)(b,\, c_{\text{out}},\, i,\, j) =
\sum_{c_{\text{in}}=0}^{C_{\text{in}}-1}
\sum_{p=0}^{H_{\text{k}}-1}
\sum_{q=0}^{W_{\text{k}}-1}
I_{\text{padded}}(b,\, c_{\text{in}},\, i \cdot S_{\text{h}} + p,\, j \cdot S_{\text{w}} + q)\,
K_{\text{dilated}}(c_{\text{out}},\, c_{\text{in}},\, p,\, q)
$$
The shape of the output is given by:
$$
(B,\, C_{\text{out}},\, H_{\text{out}},\, W_{\text{out}})
$$
- $H_{\text{out}} = \left\lfloor \dfrac{H + 2 P_\text{h} - D_\text{h} (H_\text{k} - 1) - 1}{S_\text{h}} + 1 \right\rfloor$
- $W_{\text{out}} = \left\lfloor \dfrac{W + 2 P_\text{w} - D_\text{w} (W_\text{k} - 1) - 1}{S_\text{w}} + 1 \right\rfloor$

This is the equation we use to compute the 2D convolution in practice. Next, we will discuss some different algorithms to implement the 2D convolution, as well as their pros and cons.

#### Nested for-loop
The most straightforward way to compute the 2D convolution is to use a nested for-loop. In short, this algorithm strictly follows the math equation by iterating over every batch, channel, pixel and parameter. The corresponding pseudocode is shown below:
```
for b in range(B):
    for c_out in range(C_out):
        for i in range(H_out):
            for j in range(W_out):
                sum = 0
                for c_in in range(C_in):
                    for p in range(H_k):
                        for q in range(W_k):
                            sum += I_padded[b, c_in, i * S_h + p, j * S_w + q] * K_dilated[c_out, c_in, p, q]
                output[b, c_out, i, j] = s
```
Pros:
- It is conceptually simple, clearly reflects the definition of 2D convolution, easy to understand and implement.
- It has a clear structure, can be easily modified for debugging, visualization or other purpose.

Cons:
- It is computationally expensive, especially for large input images or deep networks.
- It is extremely slow due to the seven for-loops, impractical in real life.
- Its structure can't take advantage of parallel computing using CPU or GPU, slower than other optimized algorithm with vectorized structure.
- Its time complexity given by $\mathcal{O}\big(B \cdot C_{\text{out}} \cdot C_{\text{in}} \cdot H_{\text{out}} \cdot W_{\text{out}} \cdot H_{\text{k}} \cdot W_{\text{k}}\big)$ grows rapidly with these hyperparameters.

#### Im2Col
Im2Col stands for image to column, is a widely used technique for speeding up 2D convolution by converting it into a **matrix multiplication**. As the name suggests, its core idea is to flatten the part of the input image that overlaps the kernel into a column, and kernels are flattened into rows. Then, 2D convolution is computed as a single matrix multiplication which can be highly optimized on modern hardware. For example, consider an input images $I$ of shape $(H = 5, W = 5)$ and a kernel $K$ of shape $(H_{\text{k}} = 3,\, W_{\text{k}} = 3)$:
$$
I =
\begin{bmatrix}
0 & 1 & 1 & 1 & 0 \\
0 & 1 & 0 & 1 & 0 \\
0 & 1 & 1 & 1 & 0 \\
0 & 0 & 0 & 1 & 0 \\
0 & 1 & 1 & 1 & 0
\end{bmatrix},
\quad
K =
\begin{bmatrix}
1 & 2 & 3 \\
4 & 5 & 6 \\
7 & 8 & 9
\end{bmatrix}
$$
Sliding $K$ over $I$ with a stride of $(S_{\text{h}} = 2, S_{\text{w}} = 2)$ gives $4$ parts of the image $I$:
$$
I_1 =
\begin{bmatrix}
0 & 1 & 1 \\
0 & 1 & 0 \\
0 & 1 & 1
\end{bmatrix},
\quad
I_2 =
\begin{bmatrix}
1 & 1 & 0 \\
0 & 1 & 0 \\
1 & 1 & 0
\end{bmatrix},
\quad
I_3 =
\begin{bmatrix}
0 & 1 & 1 \\
0 & 0 & 0 \\
0 & 1 & 1
\end{bmatrix},
\quad
I_4 =
\begin{bmatrix}
1 & 1 & 0 \\
0 & 1 & 0 \\
1 & 1 & 0
\end{bmatrix}
$$
Then the $4$ matrices of shape $(3,\, 3)$ are flattened into $4$ column vectors of length $9$ and are stacked into a new matrix of shape $(9,\, 4)$. The kernel $K$ is flattened into a row vector of length $9$:
$$
K_{\text{new}} =
\begin{bmatrix}
1 & 2 & 3 & 4 & 5 & 6 & 7 & 8 & 9
\end{bmatrix},
\quad
I_{\text{new}} =
\begin{bmatrix}
0 & 1 & 0 & 1 \\
1 & 1 & 1 & 1 \\
1 & 0 & 1 & 0 \\
0 & 0 & 0 & 0 \\
1 & 1 & 0 & 1 \\
0 & 0 & 0 & 0 \\
0 & 1 & 0 & 1 \\
1 & 1 & 1 & 1 \\
1 & 0 & 1 & 0
\end{bmatrix}
$$
Matrix multiplication between $K_{\text{new}}$ and $I_{\text{new}}$ gives a row vector of length $4$:
$$
K_{\text{new}}I_{\text{new}} =
\begin{bmatrix}
27 & 23 & 22 & 23
\end{bmatrix}
$$
Finally, since the expected output shape is $
\left(
\dfrac{H - H_{\text{k}}}{S_{\text{h}}} + 1 = 2,\,
\dfrac{W - W_{\text{k}}}{S_{\text{w}}} + 1 = 2
\right)
$, the row vector needs to be reshaped back to $(2,\, 2)$:
$$
\text{output} =
\begin{bmatrix}
27 & 23 \\
22 & 23
\end{bmatrix}
$$
This is a simple example with 2D input. In practice, consider a batched input images $I$ of shape $(B,\, C_{\text{in}},\, H,\, W)$ and kernels $K$ of shape $(C_{\text{out}},\, C_{\text{in}},\, H_{\text{k}},\, W_{\text{k}})$, the output shape is then given by $(B,\, C_{\text{out}},\, H_{\text{out}},\, W_{\text{out}})$. First, $I$ is converted to $I_{\text{new}}$ of shape $(B,\, H_{\text{out}},\, W_{\text{out}},\, C_{\text{in}} \cdot H_{\text{k}} \cdot W_{\text{k}})$ and $K$ is converted to $K_{\text{new}}$ of shape $(C_{\text{out}},\, C_{\text{in}} \cdot H_{\text{k}} \cdot W_{\text{k}})$. Then, matrix multiplication $I_{\text{new}} K_{\text{new}}^{\mathsf{T}}$ gives a new tensor of shape $(B,\, H_{\text{out}},\, W_{\text{out}},\, C_{\text{out}})$. Finally, the new tensor needs to be permuted back to $(B,\, C_{\text{out}},\, H_{\text{out}},\, W_{\text{out}})$ to produce the output.

The Im2Col algorithm is used in our custom implementation of 2D convolution. To demonstrate our implementation, consider the following setup:
- Shape of $I$: $(B = 4,\, C_{\text{in}} = 3,\, H = 10,\, W = 10)$
- Shape of $K$: $(C_{\text{out}} = 2,\, C_{\text{in}} = 3,\, H_{\text{k}} = 5 ,\, W_{\text{k}} = 5)$
- Stride: $(S_{\text{h}} = 2,\, S_{\text{w} = 2})$
- Padding: $(P_{\text{h}} = 1,\, P_{\text{w} = 1})$
- Dilation: $(D_{\text{h}} = 2,\, D_{\text{w} = 2})$

We run our `MyConv2d.forward` and compare its output with the result obtained from `torch.nn.functional.conv2d`:

In [27]:
torch.manual_seed(42)
I_shape = (4, 3, 10, 10)
K_shape = (2, 3, 5, 5)
stride = (2, 2)
padding = (1, 1)
dilation = (2, 2)
I = torch.randint(0, 3, size=I_shape)
K = torch.randint(0, 3, size=K_shape)
torch_output = torch.nn.functional.conv2d(I, K, stride=stride, padding=padding, dilation=dilation)
output = MyConv2d.forward(Tensor(I.flatten().tolist(), shape=I_shape), 
                           Tensor(K.flatten().tolist(), shape=K_shape), stride=stride, padding=padding, dilation=dilation)
print("=== MyConv2d.forward ===")
print(output.tolist())
print("\n=== torch.nn.functional.conv2d ===")
print(torch_output.tolist())


=== MyConv2d.forward ===
[[[[53, 63], [69, 81]], [[44, 49], [53, 61]]], [[[48, 53], [64, 63]], [[36, 50], [40, 54]]], [[[67, 83], [73, 98]], [[49, 58], [67, 75]]], [[[62, 67], [84, 93]], [[42, 45], [52, 65]]]]

=== torch.nn.functional.conv2d ===
[[[[53, 63], [69, 81]], [[44, 49], [53, 61]]], [[[48, 53], [64, 63]], [[36, 50], [40, 54]]], [[[67, 83], [73, 98]], [[49, 58], [67, 75]]], [[[62, 67], [84, 93]], [[42, 45], [52, 65]]]]
